# Análisis Descriptivo de Micronegocios en Cesar
**Fuente:** DANE – Encuesta de Micronegocios (EMICRON) – Módulo características del micronegocio
**Departamento:** Cesar (COD_DEPTO = 20)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})

RUTA = (r'C:\Users\Usuario\OneDrive - Global Green Growth Institute'
        r'\Documentos\2025\Adapta Cesar\micronegocios_cesar.csv')
df_raw = pd.read_csv(RUTA, encoding='latin-1', low_memory=False)
print(f'Filas totales: {len(df_raw):,}  |  Columnas: {df_raw.shape[1]}')
cesar = df_raw[df_raw['COD_DEPTO'] == 20].copy()
universo = cesar['F_EXP'].sum()
print(f'Registros muestra Cesar: {len(cesar):,}')
print(f'Universo estimado de micronegocios en Cesar: {universo:,.0f}')

## 1. Diccionario de variables (Módulo características del micronegocio)

In [ ]:
diccionario = {
    'P1633':    'Registro en RUT (1=Sí, 2=No)',
    'P986':     'Régimen tributario (1=Simplificado, 2=Común/ordinario, 9=No sabe)',
    'P640':     'Lleva registros contables (1=Sí, 2=No)',
    'P4000':    'Razón principal para no llevar registro contable (1-8)',
    'P1055':    'Matriculado en Cámara de Comercio (1=Sí, 2=No)',
    'P1056':    'Año de matrícula en Cámara de Comercio',
    'P661':     'Matrícula mercantil renovada (1=Sí, 2=No)',
    'P1057':    'Razón principal para no estar matriculado (1-8)',
    'P4004':    'Año de vencimiento de la matrícula',
    'P2991':    'Presentó declaración de renta último año (1=Sí, 2=No)',
    'P2992':    'Presentó declaración de IVA último año (1=Sí, 2=No)',
    'P2993':    'Presentó declaración de ICA último año (1=Sí, 2=No)',
    'CLASE_TE': 'Tipo establecimiento (1=Vivienda,2=Local,3=Vía pública,4=Obra,5=Vehículo,6=Otro)',
    'AREA':     'Área geográfica (1=Cabecera, 2=Centros poblados, 3=Rural disperso)',
    'F_EXP':    'Factor de expansión (peso muestral)'
}
pd.DataFrame.from_dict(diccionario, orient='index', columns=['Descripción']).rename_axis('Variable')

## 2. Distribución por zona geográfica (AREA)

In [ ]:
zona_map = {1: 'Cabecera', 2: 'Centros poblados', 3: 'Rural disperso'}
zona = cesar.groupby('AREA')['F_EXP'].sum().rename(index=zona_map)
zona_pct = (zona / zona.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(zona.index, zona.values, color=['#2196F3', '#4CAF50', '#FF9800'])
ax.bar_label(bars, labels=[f'{v:,.0f} ({p}%)' for v, p in zip(zona.values, zona_pct.values)], padding=5)
ax.set_xlabel('Micronegocios estimados')
ax.set_title('Micronegocios por zona geográfica – Cesar')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout(); plt.show()
print(zona.to_frame('Universo').assign(Pct=zona_pct))

## 3. Formalización tributaria – RUT (P1633) y Régimen (P986)

In [ ]:
rut_map = {1: 'Sí tiene RUT', 2: 'No tiene RUT'}
reg_map = {1: 'Simplificado', 2: 'Común/Ordinario', 9: 'No sabe'}
rut = cesar.groupby('P1633')['F_EXP'].sum().rename(index=rut_map)
reg = cesar[cesar['P1633'] == 1].groupby('P986')['F_EXP'].sum().rename(index=reg_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].pie(rut.values, labels=rut.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Registro en RUT (P1633)')
axes[1].pie(reg.values, labels=reg.index, autopct='%1.1f%%',
            colors=['#2196F3', '#FF9800', '#9E9E9E'], startangle=90)
axes[1].set_title('Régimen tributario con RUT (P986)')
plt.suptitle('Formalización tributaria – Micronegocios Cesar', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('RUT:', rut.to_frame('Universo').assign(Pct=(rut/rut.sum()*100).round(1)))
print('\nRégimen (con RUT):', reg.to_frame('Universo').assign(Pct=(reg/reg.sum()*100).round(1)))

## 4. Registros contables (P640) y razón para no llevarlos (P4000)

In [ ]:
cont_map = {1: 'Lleva registros', 2: 'No lleva registros'}
razon_cont_map = {1:'No lo considera necesario',2:'No sabe cómo',3:'No tiene tiempo',
                  4:'Es muy costoso',5:'No lo exigen',6:'Negocio muy pequeño',
                  7:'Lo hace mentalmente',8:'Otro'}
cont = cesar.groupby('P640')['F_EXP'].sum().rename(index=cont_map)
razon_no = cesar[cesar['P640'] == 2].groupby('P4000')['F_EXP'].sum().rename(index=razon_cont_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(cont.values, labels=cont.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('¿Lleva registros contables? (P640)')
rs = razon_no.sort_values(ascending=True)
axes[1].barh(rs.index, rs.values, color='#FF9800')
axes[1].bar_label(axes[1].containers[0], labels=[f'{v:,.0f}' for v in rs.values], padding=4)
axes[1].set_xlabel('Micronegocios estimados')
axes[1].set_title('Razón para no llevar contabilidad (P4000)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.suptitle('Registros contables – Micronegocios Cesar', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Cámara de Comercio (P1055, P661, P1057)

In [ ]:
camara_map = {1: 'Matriculado', 2: 'No matriculado'}
renov_map  = {1: 'Renovada', 2: 'No renovada'}
razon_cam_map = {1:'No lo considera necesario',2:'Es muy costoso',3:'No sabe cómo',
                 4:'No lo exigen',5:'Trámite difícil',6:'No tiene tiempo',
                 7:'Negocio muy pequeño',8:'Otro'}
camara = cesar.groupby('P1055')['F_EXP'].sum().rename(index=camara_map)
renov  = cesar[cesar['P1055'] == 1].groupby('P661')['F_EXP'].sum().rename(index=renov_map)
razon_cam = cesar[cesar['P1055'] == 2].groupby('P1057')['F_EXP'].sum().rename(index=razon_cam_map)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].pie(camara.values, labels=camara.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Matrícula Cámara de Comercio (P1055)')
axes[1].pie(renov.values, labels=renov.index, autopct='%1.1f%%',
            colors=['#2196F3', '#FF5722'], startangle=90)
axes[1].set_title('Matrícula renovada (P661)')
rcs = razon_cam.sort_values(ascending=True)
axes[2].barh(rcs.index, rcs.values, color='#9C27B0')
axes[2].bar_label(axes[2].containers[0], labels=[f'{v:,.0f}' for v in rcs.values], padding=4)
axes[2].set_xlabel('Micronegocios estimados')
axes[2].set_title('Razón para no matricularse (P1057)')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.suptitle('Cámara de Comercio – Micronegocios Cesar', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Declaraciones tributarias: Renta, IVA e ICA (P2991–P2993)

In [ ]:
tribut_vars = {'P2991': 'Renta', 'P2992': 'IVA', 'P2993': 'ICA'}
resumen = []
for var, etiqueta in tribut_vars.items():
    si  = cesar[cesar[var] == 1]['F_EXP'].sum()
    no  = cesar[cesar[var] == 2]['F_EXP'].sum()
    tot = si + no
    resumen.append({'Declaración': etiqueta, 'Sí presentó': si, 'No presentó': no, '% Sí': round(si/tot*100,1)})
df_trib = pd.DataFrame(resumen).set_index('Declaración')
print(df_trib)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df_trib)); w = 0.35
b1 = ax.bar(x - w/2, df_trib['Sí presentó'], w, label='Sí presentó', color='#4CAF50')
b2 = ax.bar(x + w/2, df_trib['No presentó'], w, label='No presentó', color='#F44336')
ax.set_xticks(x); ax.set_xticklabels(df_trib.index)
ax.set_ylabel('Micronegocios estimados')
ax.set_title('Declaraciones tributarias – Micronegocios Cesar')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.bar_label(b1, labels=[f'{v:,.0f}' for v in df_trib['Sí presentó']], rotation=45, padding=3, fontsize=9)
ax.bar_label(b2, labels=[f'{v:,.0f}' for v in df_trib['No presentó']], rotation=45, padding=3, fontsize=9)
ax.legend(); plt.tight_layout(); plt.show()

## 7. Índice sintético de formalización (3 dimensiones)

In [ ]:
cesar['formal_rut']    = (cesar['P1633'] == 1).astype(int)
cesar['formal_cont']   = (cesar['P640']  == 1).astype(int)
cesar['formal_camara'] = (cesar['P1055'] == 1).astype(int)
cesar['indice_formal'] = cesar[['formal_rut','formal_cont','formal_camara']].sum(axis=1)

idx_dist = cesar.groupby('indice_formal')['F_EXP'].sum()
idx_pct  = (idx_dist / idx_dist.sum() * 100).round(1)
etiq = ['0 – Sin formalización','1 – Una dimensión','2 – Dos dimensiones','3 – Completamente formal']
cols = ['#F44336','#FF9800','#2196F3','#4CAF50']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(etiq[:len(idx_dist)], idx_dist.values, color=cols[:len(idx_dist)])
ax.bar_label(bars, labels=[f'{v:,.0f}\n({p}%)' for v, p in zip(idx_dist.values, idx_pct.values)], padding=4)
ax.set_ylabel('Micronegocios estimados')
ax.set_title('Índice sintético de formalización (0–3) – Cesar')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.show()

def tasa(df, var):
    return df[df[var] == 1]['F_EXP'].sum() / df['F_EXP'].sum() * 100

print('Tasa de formalización por dimensión:')
for var, nombre in [('formal_rut','RUT'),('formal_cont','Contabilidad'),('formal_camara','Cámara de Comercio')]:
    print(f'  {nombre}: {tasa(cesar, var):.1f}%')

## 8. Tipo de establecimiento (CLASE_TE)

In [ ]:
clase_map = {1:'Vivienda/parte de vivienda',2:'Local/oficina/bodega',
             3:'Vía pública/espacio abierto',4:'En obra/construcción',5:'Vehículo',6:'Otro'}
clase = cesar.groupby('CLASE_TE')['F_EXP'].sum().rename(index=clase_map).sort_values(ascending=False)
clase_pct = (clase / clase.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(clase.index[::-1], clase.values[::-1], color='#00897B')
ax.bar_label(bars, labels=[f'{v:,.0f} ({p}%)'
             for v, p in zip(clase.values[::-1], clase_pct.values[::-1])], padding=5)
ax.set_xlabel('Micronegocios estimados')
ax.set_title('Tipo de establecimiento – Micronegocios Cesar (CLASE_TE)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.tight_layout(); plt.show()

## 9. Comparación Cesar vs. total nacional

In [ ]:
nacional = df_raw.copy()
nacional['formal_rut']    = (nacional['P1633'] == 1).astype(int)
nacional['formal_cont']   = (nacional['P640']  == 1).astype(int)
nacional['formal_camara'] = (nacional['P1055'] == 1).astype(int)

comparacion = pd.DataFrame({
    'Cesar':    [tasa(cesar,   'formal_rut'), tasa(cesar,   'formal_cont'), tasa(cesar,   'formal_camara')],
    'Nacional': [tasa(nacional,'formal_rut'), tasa(nacional,'formal_cont'), tasa(nacional,'formal_camara')]
}, index=['RUT','Registros contables','Cámara de Comercio']).round(1)
print(comparacion)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(comparacion)); w = 0.35
b1 = ax.bar(x - w/2, comparacion['Cesar'],    w, label='Cesar',    color='#2196F3')
b2 = ax.bar(x + w/2, comparacion['Nacional'], w, label='Nacional', color='#9E9E9E')
ax.set_xticks(x); ax.set_xticklabels(comparacion.index)
ax.set_ylabel('Tasa de formalización (%)')
ax.set_title('Formalización: Cesar vs. Nacional')
ax.bar_label(b1, labels=[f'{v:.1f}%' for v in comparacion['Cesar']],    padding=3)
ax.bar_label(b2, labels=[f'{v:.1f}%' for v in comparacion['Nacional']], padding=3)
ax.set_ylim(0, 100); ax.legend(); plt.tight_layout(); plt.show()

## 10. Resumen ejecutivo

In [ ]:
univ       = cesar['F_EXP'].sum()
pct_rut    = tasa(cesar, 'formal_rut')
pct_cont   = tasa(cesar, 'formal_cont')
pct_camara = tasa(cesar, 'formal_camara')
pct_3dim   = cesar[cesar['indice_formal'] == 3]['F_EXP'].sum() / univ * 100
pct_0dim   = cesar[cesar['indice_formal'] == 0]['F_EXP'].sum() / univ * 100
pct_renta  = (cesar[cesar['P2991'] == 1]['F_EXP'].sum()
              / cesar[cesar['P2991'].isin([1,2])]['F_EXP'].sum() * 100)

print('=' * 62)
print('RESUMEN EJECUTIVO – MICRONEGOCIOS CESAR (EMICRON DANE)')
print('=' * 62)
print(f'Universo estimado de micronegocios:  {univ:>12,.0f}')
print('-' * 62)
print('DIMENSIÓN TRIBUTARIA')
print(f'  Con RUT registrado:                {pct_rut:>11.1f}%')
print(f'  Declararon renta (último año):     {pct_renta:>11.1f}%')
print('-' * 62)
print('DIMENSIÓN CONTABLE')
print(f'  Llevan registros contables:        {pct_cont:>11.1f}%')
print('-' * 62)
print('DIMENSIÓN LEGAL')
print(f'  Matriculados Cámara de Comercio:   {pct_camara:>11.1f}%')
print('-' * 62)
print('ÍNDICE SINTÉTICO (3 dimensiones)')
print(f'  Completamente formales (3/3):      {pct_3dim:>11.1f}%')
print(f'  Completamente informales (0/3):    {pct_0dim:>11.1f}%')
print('=' * 62)

---
# PARTE 2 – Movilización FNG por CIIU: concentración nacional (2024–2025)
**Fuente:** Movilización_a_Actividades_Sostenibles.xlsx  
**Nivel:** Nacional — No. de operaciones y valor movilizado por sector económico

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

RUTA_FNG = (r'C:\Users\Usuario\OneDrive - Global Green Growth Institute'
            r'\Documentos\2025\Outputs\Output5\3. Clasificación de destinos FNG'
            r'\Movilización_a_Actividades_Sostenibles.xlsx')

raw = pd.ExcelFile(RUTA_FNG, engine='openpyxl').parse('2024-2025_CIIU', header=None)
df_mov = raw.iloc[7:].copy()
df_mov.columns = ['Producto','Nombre_Producto','CIIU','Gar2024','Val2024','Gar2025','Val2025']
df_mov['CIIU'] = pd.to_numeric(df_mov['CIIU'], errors='coerce')
df_mov = df_mov.dropna(subset=['CIIU'])
df_mov['CIIU'] = df_mov['CIIU'].astype(int).astype(str).str.zfill(4)
for c in ['Gar2024','Val2024','Gar2025','Val2025']:
    df_mov[c] = pd.to_numeric(df_mov[c], errors='coerce').fillna(0)

ciiu = df_mov.groupby('CIIU')[['Gar2024','Val2024','Gar2025','Val2025']].sum()
ciiu['Gar_total'] = ciiu['Gar2024'] + ciiu['Gar2025']
ciiu['Val_total'] = ciiu['Val2024'] + ciiu['Val2025']
ciiu['Val_B']     = ciiu['Val_total'] / 1e9

print(f"CIIU activos: {len(ciiu):,}  |  "
      f"Operaciones totales: {ciiu['Gar_total'].sum():,.0f}  |  "
      f"Valor movilizado: ${ciiu['Val_B'].sum():,.0f} miles de millones COP")

In [ ]:
NOMBRES = {
    '0111':'Cereales (otros)',        '0112':'Arroz',
    '0113':'Maíz',                    '0119':'Otros cultivos transitorios',
    '0121':'Hortalizas',              '0122':'Legumbres secas',
    '0123':'Café',                    '0124':'Caña de azúcar',
    '0125':'Flores de corte',         '0126':'Palma de aceite',
    '0127':'Plantas oleaginosas',     '0128':'Plantas medicinales/aromáticas',
    '0141':'Ganadería bovina (carne)','0142':'Ganadería bovina (leche)',
    '0143':'Ganadería porcina',       '0144':'Avicultura postura',
    '0145':'Avicultura engorde',      '0149':'Otras actividades pecuarias',
    '0150':'Cacao',                   '0151':'Plátano / banano',
    '0152':'Aguacate',                '0153':'Cítricos',
    '0154':'Piña',                    '0155':'Papaya',
    '0156':'Mango',                   '0161':'Silvicultura',
    '0210':'Pesca y acuicultura',
    '1011':'Procesamiento cárnico',   '1040':'Aceites y grasas vegetales',
    '1061':'Molinería',               '1081':'Panadería',
    '1084':'Comidas preparadas',      '1410':'Confección de prendas',
    '3511':'Generación de energía eléctrica',
    '3600':'Captación y tratamiento de agua',
    '3700':'Evacuación de aguas residuales',
    '3811':'Recolección de residuos no peligrosos',
    '3830':'Recuperación de materiales',
    '4321':'Instalaciones eléctricas',
    '4390':'Otras actividades de construcción',
    '4520':'Mantenimiento de vehículos',
    '4711':'Comercio alimentos (supermercados)',
    '4719':'Comercio minorista no especializado',
    '4721':'Comercio frutas y verduras',
    '4723':'Comercio carnes',         '4729':'Comercio alimentos varios',
    '4759':'Comercio artículos hogar','4771':'Comercio prendas de vestir',
    '4923':'Transporte de carga por carretera',
    '5611':'Restaurantes y expendio de comidas',
    '9602':'Peluquería y cuidado personal',
    '9609':'Otros servicios personales',
}

# Solo sectores con evidencia real de actividades de mitigación o adaptación en Colombia
POTENCIAL = {
    '0123': 'Adaptación y mitigación',  # café: agroforestería, variedades resistentes
    '0150': 'Adaptación y mitigación',  # cacao: sistemas agroforestales, captura carbono
    '0126': 'Adaptación y mitigación',  # palma: RSPO, bioenergía certificada
    '0141': 'Adaptación y mitigación',  # ganadería carne: silvopastoreo, reducción metano
    '0142': 'Adaptación y mitigación',  # ganadería leche: silvopastoreo
    '0161': 'Mitigación',               # silvicultura: REDD+, captura de carbono
    '0111': 'Adaptación', '0112': 'Adaptación', '0113': 'Adaptación',
    '0119': 'Adaptación', '0121': 'Adaptación', '0122': 'Adaptación',
    '0124': 'Adaptación', '0125': 'Adaptación', '0127': 'Adaptación',
    '0151': 'Adaptación', '0152': 'Adaptación', '0153': 'Adaptación',
    '0154': 'Adaptación', '0155': 'Adaptación', '0156': 'Adaptación',
    '0143': 'Adaptación', '0144': 'Adaptación', '0145': 'Adaptación',
    '0210': 'Adaptación',               # pesca: gestión sostenible, ecosistemas
    '3511': 'Mitigación',               # generación eléctrica: crecimiento renovables
    '3600': 'Adaptación',               # agua: seguridad hídrica ante cambio climático
    '3700': 'Mitigación',               # aguas residuales: reducción CH₄ en vertimientos
    '3811': 'Mitigación',               # residuos: desvío de rellenos, reducción CH₄
    '3830': 'Mitigación',               # recuperación materiales: reciclaje
    '4321': 'Mitigación',               # instalaciones eléctricas: solar, eficiencia
    '4923': 'Mitigación',               # transporte carga: electrificación de flotas
}

ciiu['Sector']    = ciiu.index.map(lambda c: NOMBRES.get(c, f'CIIU {c}'))
ciiu['Potencial'] = ciiu.index.map(POTENCIAL)
ciiu['Es_pot']    = ciiu['Potencial'].notna()

PAL  = {'Adaptación y mitigación': '#1B5E20', 'Mitigación': '#2196F3', 'Adaptación': '#FF9800'}
GRIS = '#CFD8DC'

def color(row): return PAL.get(row['Potencial'], GRIS)

top30_gar = ciiu.nlargest(30, 'Gar_total').reset_index()
top30_val = ciiu.nlargest(30, 'Val_B').reset_index()
leg = [Patch(facecolor=c, label=s) for s,c in PAL.items()] + \
      [Patch(facecolor=GRIS, label='Sin potencial identificado')]

print(f"Sectores con potencial sostenible: {ciiu['Es_pot'].sum()} CIIU")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
cols = [color(r) for _, r in top30_gar.iterrows()]
bars = ax.barh(top30_gar['Sector'][::-1], top30_gar['Gar_total'][::-1],
               color=cols[::-1], edgecolor='white', linewidth=0.4)
ax.bar_label(bars, labels=[f'{v:,.0f}' for v in top30_gar['Gar_total'][::-1]],
             padding=4, fontsize=8)
ax.set_xlabel('Total operaciones garantizadas (2024 + 2025)')
ax.set_title('Top 30 CIIU – Operaciones FNG (nacional)\nColor = potencial de actividades sostenibles',
             fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.legend(handles=leg, loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
cols2 = [color(r) for _, r in top30_val.iterrows()]
bars2 = ax.barh(top30_val['Sector'][::-1], top30_val['Val_B'][::-1],
                color=cols2[::-1], edgecolor='white', linewidth=0.4)
ax.bar_label(bars2, labels=[f'${v:,.0f}B' for v in top30_val['Val_B'][::-1]],
             padding=4, fontsize=8)
ax.set_xlabel('Valor movilizado (miles de millones COP, 2024 + 2025)')
ax.set_title('Top 30 CIIU – Valor movilizado FNG (nacional)\nColor = potencial de actividades sostenibles',
             fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.legend(handles=leg, loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
pot = ciiu[ciiu['Es_pot']].copy().reset_index()

fig, ax = plt.subplots(figsize=(12, 7))
for _, row in pot.iterrows():
    c = PAL.get(row['Potencial'], GRIS)
    ax.scatter(row['Gar_total'], row['Val_B'], s=90, color=c,
               alpha=0.85, edgecolors='white', linewidth=0.5)
    if row['Gar_total'] > 5000 or row['Val_B'] > 150:
        ax.annotate(row['Sector'], (row['Gar_total'], row['Val_B']),
                    textcoords='offset points', xytext=(5, 3), fontsize=8)
ax.set_xlabel('Total operaciones (2024 + 2025)')
ax.set_ylabel('Valor movilizado (miles de millones COP)')
ax.set_title('Sectores con potencial sostenible — volumen vs valor (nacional)', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.legend(handles=leg, fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
total_g = ciiu['Gar_total'].sum()
total_v = ciiu['Val_B'].sum()
share = ciiu.groupby('Potencial')[['Gar_total','Val_B']].sum().reindex(PAL.keys()).dropna()
share['%_gar'] = share['Gar_total'] / total_g * 100
share['%_val'] = share['Val_B']    / total_v * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cols_s = [PAL[i] for i in share.index]
b1 = axes[0].bar(share.index, share['%_gar'], color=cols_s)
axes[0].bar_label(b1, labels=[f'{v:.1f}%\n({n:,.0f} ops)'
                               for v,n in zip(share['%_gar'], share['Gar_total'])],
                  padding=4, fontsize=9)
axes[0].set_ylabel('% del total operaciones')
axes[0].set_title('Participación en operaciones FNG')
axes[0].set_ylim(0, share['%_gar'].max() * 1.35)
axes[0].tick_params(axis='x', labelrotation=10)

b2 = axes[1].bar(share.index, share['%_val'], color=cols_s)
axes[1].bar_label(b2, labels=[f'{v:.1f}%\n(${n:,.0f}B)'
                               for v,n in zip(share['%_val'], share['Val_B'])],
                  padding=4, fontsize=9)
axes[1].set_ylabel('% del valor movilizado')
axes[1].set_title('Participación en valor movilizado FNG')
axes[1].set_ylim(0, share['%_val'].max() * 1.35)
axes[1].tick_params(axis='x', labelrotation=10)

plt.suptitle('Potencial sostenible en el portafolio FNG — nacional (2024+2025)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
print('='*68)
print('RESUMEN EJECUTIVO – FNG NACIONAL: POTENCIAL SOSTENIBLE POR CIIU')
print('='*68)
print(f'Total operaciones 2024+2025:       {total_g:>14,.0f}')
print(f'Total valor movilizado (miles MM): ${total_v:>13,.0f}B')
print(f'CIIU distintos con actividad:      {len(ciiu):>14,}')
print('-'*68)
for cat in PAL:
    g = ciiu[ciiu['Potencial']==cat]['Gar_total'].sum()
    v = ciiu[ciiu['Potencial']==cat]['Val_B'].sum()
    n = ciiu[ciiu['Potencial']==cat].shape[0]
    print(f'{cat:<30}  {n:>3} CIIU  {g:>9,.0f} ops ({g/total_g*100:.1f}%)  ${v:,.0f}B ({v/total_v*100:.1f}%)')
print('-'*68)
print('Top 8 por operaciones (potencial sostenible):')
top8 = ciiu[ciiu['Es_pot']].nlargest(8,'Gar_total')[['Sector','Potencial','Gar_total','Val_B']]
for _,r in top8.iterrows():
    print(f'  {r.Sector:<40} [{r.Potencial}]  {r.Gar_total:>8,.0f} ops  ${r.Val_B:,.0f}B')
print('='*68)

---
# PARTE 3 – Lectura de datos climáticos (NetCDF)
## Precipitación mensual acumulada 1981–2023
**Fuente:** Grillas NetCDF – `prcp_mes_acum_1981_2023`

In [ ]:
import os
import xarray as xr

# Directorio con los archivos NetCDF mensuales
RUTA_NC = (r'C:\Users\Usuario\OneDrive - Global Green Growth Institute'
           r'\Documentos\2025\Outputs\Output4\prcp_mes_acum_1981_2023')

# Listar archivos disponibles
archivos = sorted([f for f in os.listdir(RUTA_NC) if f.endswith('.nc')])
print(f'Archivos disponibles: {len(archivos)}')
print('Primeros 5:', archivos[:5])

# Leer un solo archivo (sin dask)
archivo_ejemplo = os.path.join(RUTA_NC, archivos[0])
ds = xr.open_dataset(archivo_ejemplo)
print('\n', ds)

# Acceder a la variable de precipitación
prcp = ds['prcp']
print(f'\nDimensiones: {dict(prcp.dims)}')
print(f'Lat: {float(prcp.Lat.min()):.1f} a {float(prcp.Lat.max()):.1f}')
print(f'Lon: {float(prcp.Lon.min()):.1f} a {float(prcp.Lon.max()):.1f}')
print(f'Precipitación máxima en el archivo: {float(prcp.max()):.1f} mm')